# Gate 5: held-out rewrites (run once)

Rules: `docs/ROADMAP.md`, gate 5. The three `qlora_v1` adapters from gate 4b (on Google Drive, not retrained) are
merged into their bf16 bases and rewrite the 20 held-out questions (kvkk-rag `eval/test_v2.jsonl`) with exactly
the gate 4b inference. Retrieval is scored locally afterwards with `scripts/eval_rewrites.py --set heldout`.

The held-out set is run once. If the runtime disconnects, run all again: models whose output is already on Drive
are skipped, and a finished model is never regenerated.

Before running:
1. Runtime -> Change runtime type -> **L4 GPU**.
2. Secrets (key icon) -> `HF_TOKEN` (Hugging Face read token with Llama-3.1 access), notebook access on.
3. Runtime -> Run all, and allow Google Drive access when asked (the same account as gate 4b). Keep the tab open.
4. At the end a zip with the result files downloads. Add the row printed by the last cell to
   `results/colab_usage.csv`, then disconnect and delete the runtime.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert "L4" in gpu, f"expected an L4 runtime (docs/ROADMAP.md, gate 5), got {gpu!r}"

In [ ]:
import os
if not os.path.exists("/content/kvkk-rewriter"):
    subprocess.run(["git", "clone", "-q", "--recurse-submodules",
                    "https://github.com/EmirhanHasirci11/kvkk-rewriter.git", "/content/kvkk-rewriter"], check=True)
%cd /content/kvkk-rewriter
!git pull -q && git log -1 --format='%H %s'
!pip install -q "transformers>=4.56" accelerate peft bitsandbytes
# Colab's preinstalled torchao is older than peft accepts, and loading the adapter then fails; nothing here uses it.
!pip uninstall -q -y torchao

In [ ]:
import json, sys, time
from pathlib import Path

import torch, transformers, peft, bitsandbytes
from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/kvkk-rewriter")

sys.path.insert(0, "scripts")
import qlora, student

TAG = "qlora_v1"
ORDER = ["kumru-2b", "qwen3-4b", "llama3.1-8b"]    # fastest first
HF_TOKEN = userdata.get("HF_TOKEN")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
heldout = [json.loads(l) for l in Path("external/kvkk-rag/eval/test_v2.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
assert len(heldout) == 20, len(heldout)
warmup = qlora.load_pairs("train")[0]["question"]
LOG = DRIVE / f"results/colab_{TAG}_heldout.log"
versions = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
            "bitsandbytes": bitsandbytes.__version__}

# the adapters must be the gate 4b ones: their training records equal the committed copies
for name in ORDER:
    on_drive = json.loads((DRIVE / f"adapters/{TAG}_{name}/train_record.json").read_text(encoding="utf-8"))
    committed = json.loads(Path(f"results/train_record_{TAG}_{name}.json").read_text(encoding="utf-8"))
    assert on_drive == committed, f"{name}: adapter on Drive does not match results/train_record_{TAG}_{name}.json"


def log(msg):
    line = f"{time.strftime('%H:%M:%S')} {msg}"
    print(line)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")


log(f"start: {gpu}, commit {COMMIT[:7]}, {versions}")
print(len(heldout), "held-out questions; adapters checked")

In [ ]:
t_start = time.time()
for name in ORDER:
    hf_id = student.MODELS[name]
    adapter = DRIVE / f"adapters/{TAG}_{name}"
    out = DRIVE / f"results/rewrites_{TAG}_{name}_heldout_v1.jsonl"
    if out.exists():
        log(f"=== {name}: {out.name} already on Drive, skipped")
        continue
    log(f"=== {name}: held-out rewrites ({hf_id} + {adapter.name})")
    model, tok, stop = qlora.load_merged(hf_id, adapter, token=HF_TOKEN)
    torch.cuda.reset_peak_memory_stats()
    student.rewrite(model, tok, stop, warmup)
    rows = []
    for i, q in enumerate(heldout, 1):
        r = student.rewrite(model, tok, stop, q["question"])
        rows.append({"id": q["id"], "question": q["question"], **r, "model": name, "hf_id": hf_id, "adapter": adapter.name})
        print(f"[{i:2}/{len(heldout)}] {r['latency_s']:.2f}s {r['new_tokens']:3} tok{' TRUNC' if r['truncated'] else ''}  {r['rewrite'][:110]}")
    out.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
    lat = sorted(r["latency_s"] for r in rows)
    log(f"{name}: held-out rewrites written; latency p50 {lat[len(lat) // 2]} s, truncated {sum(r['truncated'] for r in rows)}, "
        f"inference peak {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    del model
    torch.cuda.empty_cache()
log(f"done in {(time.time() - t_start) / 60:.1f} min this session")

In [ ]:
import zipfile
from google.colab import files
paths = sorted((DRIVE / "results").glob(f"rewrites_{TAG}_*_heldout_v1.jsonl")) + [LOG]
with zipfile.ZipFile(f"{TAG}_heldout.zip", "w") as z:
    for p in paths:
        z.write(p, f"results/{p.name}")
files.download(f"{TAG}_heldout.zip")
print("row for results/colab_usage.csv (fill in runtime_min and compute_units from Colab):")
print(f"{time.strftime('%Y-%m-%d')},notebooks/03_heldout.ipynb,{gpu},<runtime_min>,<compute_units>,held-out rewrites ({len(paths) - 1} models),commit {COMMIT[:7]}")